In [12]:
%%writefile softmax_check.cu
#include <cuda_bf16.h>
#include <cuda_fp16.h>
#include <cuda_fp8.h>
#include <cuda_runtime.h>
#include <float.h>
#include <math.h>
#include <stdio.h>
#include <stdlib.h>
#include <chrono>
#include <iostream>
#include <random>

#define WARP_SIZE 32

#define CUDA_CHECK(call)                                                      \
    do {                                                                      \
        cudaError_t err = (call);                                             \
        if (err != cudaSuccess) {                                             \
            std::cerr << "CUDA error: " << cudaGetErrorString(err)            \
                      << " at " << __FILE__ << ":" << __LINE__ << std::endl; \
            std::exit(1);                                                     \
        }                                                                     \
    } while (0)

template <const int kWarpSize = WARP_SIZE>
__device__ __forceinline__ float warp_reduce_sum_f32(float val) {
    #pragma unroll
    for (int mask = kWarpSize >> 1; mask >= 1; mask >>= 1) {
        val += __shfl_xor_sync(0xffffffff, val, mask);
    }
    return val;
}

template <const int NUM_THREADS = 256>
__device__ __forceinline__ float block_reduce_sum_f32(float val) {
    int lane = threadIdx.x % WARP_SIZE;
    int warp = threadIdx.x / WARP_SIZE;
    constexpr int NUM_WARPS = (NUM_THREADS + WARP_SIZE - 1) / WARP_SIZE;

    static __shared__ float reduce_smem[NUM_WARPS];

    float sum = warp_reduce_sum_f32<WARP_SIZE>(val);

    if (lane == 0) {
        reduce_smem[warp] = sum;
    }
    __syncthreads();

    sum = (lane < NUM_WARPS) ? reduce_smem[lane] : 0.0f;
    sum = warp_reduce_sum_f32<NUM_WARPS>(sum);
    sum = __shfl_sync(0xffffffff, sum, 0);
    return sum;
}

template <int NUM_THREADS = 256>
__global__ void softmax_f32_per_token(float* A, float* C, int N) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    float exp_val = (idx < N) ? expf(A[idx]) : 0.0f;
    float exp_sum = block_reduce_sum_f32<NUM_THREADS>(exp_val);

    if (idx < N) {
        C[idx] = exp_val / exp_sum;
    }
}

void softmax_cpu_naive_1d(const float* A, float* C, int N) {
    double denom = 0.0;
    for (int i = 0; i < N; i++) {
        denom += (double)expf(A[i]);
    }
    float d = (float)denom;
    for (int i = 0; i < N; i++) {
        C[i] = expf(A[i]) / d;
    }
}

static double wall_ms_now() {
    using clock = std::chrono::high_resolution_clock;
    return std::chrono::duration<double, std::milli>(
        clock::now().time_since_epoch()).count();
}

double measure_cpu_ms(int iters, const float* A, float* C, int N) {
    double t0 = wall_ms_now();
    for (int i = 0; i < iters; i++) {
        softmax_cpu_naive_1d(A, C, N);
    }
    double t1 = wall_ms_now();
    return (t1 - t0) / iters;
}

float measure_gpu_ms(int iters, float* dA, float* dC, int N) {
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    dim3 block(256);
    dim3 grid(1);

    softmax_f32_per_token<256><<<grid, block>>>(dA, dC, N);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaEventRecord(start));
    for (int i = 0; i < iters; i++) {
        softmax_f32_per_token<256><<<grid, block>>>(dA, dC, N);
    }
    CUDA_CHECK(cudaEventRecord(stop));
    CUDA_CHECK(cudaEventSynchronize(stop));

    float ms = 0.0f;
    CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));

    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));
    return ms / iters;
}

bool compare_results(const float* ref, const float* out, int N, float atol, float rtol) {
    double max_abs = 0.0;
    double max_rel = 0.0;
    for (int i = 0; i < N; i++) {
        double a = ref[i];
        double b = out[i];
        double abs_err = fabs(a - b);
        double rel_err = abs_err / (fabs(a) + 1e-12);
        if (abs_err > max_abs) max_abs = abs_err;
        if (rel_err > max_rel) max_rel = rel_err;
        if (!(abs_err <= atol + rtol * fabs(a))) {
            std::cerr << "Mismatch at " << i << " ref=" << a << " out=" << b << std::endl;
            return false;
        }
    }
    std::cout << "Max abs err: " << max_abs << std::endl;
    std::cout << "Max rel err: " << max_rel << std::endl;
    return true;
}

int main() {
    int N = 256; // must be <= 256 for this kernel
    size_t bytes = N * sizeof(float);

    float* A_host = (float*)malloc(bytes);
    float* C_host_cpu = (float*)malloc(bytes);
    float* C_host_gpu = (float*)malloc(bytes);

    std::mt19937 rng(0);
    std::uniform_real_distribution<float> dist(-2.0f, 2.0f);
    for (int i = 0; i < N; i++) {
        A_host[i] = dist(rng);
    }

    double cpu_ms = measure_cpu_ms(1000, A_host, C_host_cpu, N);
    std::cout << "CPU time: " << cpu_ms << " ms" << std::endl;

    float *A_device, *C_device;
    CUDA_CHECK(cudaMalloc(&A_device, bytes));
    CUDA_CHECK(cudaMalloc(&C_device, bytes));
    CUDA_CHECK(cudaMemcpy(A_device, A_host, bytes, cudaMemcpyHostToDevice));

    float gpu_ms = measure_gpu_ms(10000, A_device, C_device, N);
    std::cout << "GPU time: " << gpu_ms << " ms" << std::endl;
    std::cout << "Speedup: " << cpu_ms / gpu_ms << "x" << std::endl;

    CUDA_CHECK(cudaMemcpy(C_host_gpu, C_device, bytes, cudaMemcpyDeviceToHost));

    bool ok = compare_results(C_host_cpu, C_host_gpu, N, 1e-6f, 1e-5f);
    std::cout << (ok ? "Results match!" : "Results do not match!") << std::endl;

    return ok ? 0 : 1;
}

Overwriting softmax_check.cu


In [14]:
%%bash
set -e
nvcc -O3 -std=c++17 \
  -gencode arch=compute_75,code=sm_75 \
  softmax_check.cu -o softmax_check
./softmax_check

CPU time: 0.00220142 ms
GPU time: 0.00405984 ms
Speedup: 0.542242x
Max abs err: 1.86265e-09
Max rel err: 2.34131e-07
Results match!
